# 6. Evaluating pathways

**Learning objective:** understand what this project checks automatically, what
it deliberately leaves to a person, and how to read a failing check as a signal
about which part of the system is broken.

**Where this fits:** evaluation wraps around everything built so far.

```
retrieval evaluation  -> is the right evidence coming back?
pathway evaluation    -> is the generated pathway safe, grounded, and personal?
human review          -> is it actually good?
```

Everything in this notebook runs offline against synthetic pathways. No Azure
call is made.

## Two evaluations, two questions

Retrieval evaluation and pathway evaluation answer different questions, and
mixing them makes both useless.

Retrieval evaluation asks whether the evidence was available. It needs a query
set with known correct sources, and it reports hit rate and mean reciprocal rank.

Pathway evaluation asks whether the generated text obeys the rules a family-facing
document must obey. It needs no ground truth, because every rule is checked
against the pathway and the evidence that produced it.

In [ ]:
from mosaic_pathway.models import (
    ChildProfile,
    CommunitySuggestion,
    EvaluationCase,
    FamilyIntake,
    GroundedPathwayResult,
    LearningPathway,
    ResourceRecommendation,
    RetrievedRecord,
    RhythmPractice,
    SourceRecord,
)
from mosaic_pathway.pathway_evaluation import (
    PROHIBITED_PHRASES,
    evaluate_grounded_pathway,
    mentions_any,
    pathway_text,
    summarize_evaluations,
)
from mosaic_pathway.retrieval_evaluation import QueryEvaluation, summarize

print("prohibited phrases tracked:", len(PROHIBITED_PHRASES))

In [ ]:
retrieval_evaluations = [
    QueryEvaluation(
        query_id="outdoor-rhythm",
        query="a family wanting more outdoor time",
        expected_source_ids=["synthetic-guide"],
        retrieved_source_ids=["synthetic-guide", "synthetic-other"],
        first_hit_rank=1,
    ),
    QueryEvaluation(
        query_id="interest-led",
        query="following a child's interest in animals",
        expected_source_ids=["synthetic-guide"],
        retrieved_source_ids=["synthetic-other", "synthetic-guide"],
        first_hit_rank=2,
    ),
    QueryEvaluation(
        query_id="family-criticism",
        query="handling criticism from extended family",
        expected_source_ids=["synthetic-support"],
        retrieved_source_ids=["synthetic-other", "synthetic-guide"],
        first_hit_rank=None,
    ),
]

retrieval_summary = summarize(retrieval_evaluations)

print("hit rate at k       :", round(retrieval_summary.hit_rate_at_k, 3))
print("mean reciprocal rank:", round(retrieval_summary.mean_reciprocal_rank, 3))

## Cases are fixtures, not runs

A case is a saved family intake with an id and a description. Keeping cases as
files means the same families are evaluated every time, so a change in results
is a change in the system rather than a change in the input.

The six real cases cover a range the project cares about: a single younger child,
two children of different ages, a neurodivergent teenager, tight practical
constraints, and families at different stages of leaving a school system.

In [ ]:
intake = FamilyIntake(
    children=[
        ChildProfile(
            label="older child",
            age=12,
            interests=["animals", "drawing"],
            learning_needs=["movement breaks"],
        )
    ],
    leaving_behind=["rigid daily schedules"],
    wants_to_preserve=["reading together after dinner"],
    wants_to_add=["more time outdoors"],
    family_values=["curiosity", "gentleness"],
    practical_constraints=["one working parent at home"],
)

case = EvaluationCase(
    case_id="synthetic-curious-twelve",
    description="One twelve year old leaving a rigid timetable behind.",
    intake=intake,
)

print(case.case_id, "|", case.description)

In [ ]:
def make_record(index: int, text: str) -> SourceRecord:
    return SourceRecord(
        source_id=f"synthetic-guide-{index:04d}",
        title=f"Synthetic guide chunk {index}",
        source_file="synthetic-guide.docx",
        content_type="practical_guidance",
        authority_type="mosaic_guidance",
        topics=["rhythm"],
        text=f"A synthetic passage used only for evaluation demonstrations. {text}",
    )


RETRIEVED = [
    RetrievedRecord(record=make_record(7, "On starting small."), score=0.88),
    RetrievedRecord(record=make_record(11, "On interest catalogs."), score=0.81),
    RetrievedRecord(record=make_record(19, "On walking first."), score=0.77),
]

BASE_PATHWAY = LearningPathway(
    family_reflection=(
        "Your older child loves animals and drawing, and your family is trading a "
        "rigid timetable for a rhythm that protects curiosity and time outdoors."
    ),
    starting_rhythm=[
        RhythmPractice(
            timing="Most mornings",
            practice="Take a short walk before the day begins.",
            why_it_fits="It adds outdoor time without adding a schedule.",
        ),
        RhythmPractice(
            timing="Once a week",
            practice="Add one page to a shared observation journal.",
            why_it_fits="It keeps a record of what your child noticed.",
        ),
    ],
    resources=[
        ResourceRecommendation(
            title="Start an interest catalog",
            why_it_fits="It gives a growing interest somewhere to live.",
            source_id="synthetic-guide-0007",
            url=None,
        ),
        ResourceRecommendation(
            title="Build a gentle weekly rhythm",
            why_it_fits="It replaces the timetable you are leaving behind.",
            source_id="synthetic-guide-0011",
            url=None,
        ),
    ],
    community_suggestion=CommunitySuggestion(
        suggestion="Visit one informal nature meetup this month.",
        why_it_fits="It is a low pressure way to meet other families.",
        source_id="synthetic-guide-0019",
    ),
    closing_note="Go slowly. One walk and one journal page is a real start.",
)


def variant(**overrides: object) -> GroundedPathwayResult:
    """Build a grounded result whose pathway differs from the base in one way."""

    pathway = LearningPathway.model_validate(BASE_PATHWAY.model_dump() | overrides)

    return GroundedPathwayResult(
        intake=intake,
        retrieval_query="synthetic retrieval query",
        retrieved_records=RETRIEVED,
        pathway=pathway,
    )


def report(case_id: str, result: GroundedPathwayResult) -> None:
    evaluation = evaluate_grounded_pathway(case_id, result)
    status = "passed" if evaluation.passed else "FAILED"

    print(f"{case_id}: {status} ({len(evaluation.checks)} checks)")

    for check in evaluation.checks:
        if not check.passed:
            print(f"  - {check.check_name}: {check.details}")


print("evaluation fixtures ready")

## The passing case, and what the ten checks cover

The checks run in a fixed order and each returns a name, a boolean, and an
optional detail string. They fall into three groups:

| Group | Checks | What it protects |
| --- | --- | --- |
| Structure | required sections, resource count, word count guardrails, no duplicates | The document is complete and readable |
| Grounding | resource ids grounded, community id grounded, no citations in prose | Every claim traces to retrieved evidence, without exposing ids to the family |
| Tone and fit | no prohibited phrases, interest indicator, family context indicator | The pathway is safe and appears personalized |

In [ ]:
passing = variant()
report("passing-pathway", passing)

evaluation = evaluate_grounded_pathway("passing-pathway", passing)

for check in evaluation.checks:
    print(f"  {check.check_name}")

## Failure 1: a citation that was never retrieved

The most important failure in the suite. A resource cites an id that is not in
the retrieved set, so the pathway is not grounded.

In [ ]:
ungrounded_resources = [
    BASE_PATHWAY.resources[0].model_dump() | {"source_id": "synthetic-guide-0042"},
    BASE_PATHWAY.resources[1].model_dump(),
]

report("citation-failure", variant(resources=ungrounded_resources))

## Failure 2: citation leakage into the prose

Ids belong in the structured `source_id` fields. A family should never read
`[synthetic-guide-0007]` in the middle of a sentence about their child.

The check looks for bracketed markers and for any known record id appearing in
family-facing text.

In [ ]:
leaked_reflection = (
    "Your older child loves animals and drawing [synthetic-guide-0007], and your "
    "family is trading a rigid timetable for a rhythm that protects curiosity."
)

report("citation-leakage", variant(family_reflection=leaked_reflection))

## Failure 3: no lexical sign of personalization

This check is honest about being weak. It asks whether any stated interest word
appears anywhere in the pathway text. That is a keyword scan, not a judgement.

It still earns its place: a pathway that mentions none of the child's interests
is almost certainly generic, even though mentioning them proves very little.

In [ ]:
generic_reflection = (
    "Your family is trading a rigid timetable for a rhythm that protects "
    "curiosity and leaves room for unhurried mornings."
)
generic = variant(family_reflection=generic_reflection)

report("weak-personalization", generic)

print()
print(
    "interest words found in the base pathway :",
    mentions_any(pathway_text(BASE_PATHWAY), ["animals", "drawing"]),
)
print(
    "interest words found in the generic one  :",
    mentions_any(pathway_text(generic.pathway), ["animals", "drawing"]),
)

## Failure 4: prescriptive or clinical wording

Mosaic pathways are invitations, not instructions, and they are not clinical
advice. A short explicit phrase list catches the most damaging wording.

In [ ]:
print(list(PROHIBITED_PHRASES))
print()

report(
    "prohibited-phrase",
    variant(closing_note="You must keep this rhythm every day for it to work."),
)

## Aggregating a run

`summarize_evaluations` reports both case-level and check-level pass counts. The
distinction matters: one case failing one check out of ten is a very different
situation from every case failing the same check.

In [ ]:
results = [
    evaluate_grounded_pathway("passing-pathway", passing),
    evaluate_grounded_pathway(
        "citation-failure", variant(resources=ungrounded_resources)
    ),
    evaluate_grounded_pathway(
        "citation-leakage", variant(family_reflection=leaked_reflection)
    ),
    evaluate_grounded_pathway("weak-personalization", generic),
    evaluate_grounded_pathway(
        "prohibited-phrase",
        variant(closing_note="You must keep this rhythm every day for it to work."),
    ),
]

report_summary = summarize_evaluations(results)

print("cases evaluated :", report_summary.cases_evaluated)
print("cases passed    :", report_summary.cases_passed)
print("checks evaluated:", report_summary.checks_evaluated)
print("checks passed   :", report_summary.checks_passed)
print("check pass rate :", round(report_summary.check_pass_rate, 3))
print()

for result in report_summary.results:
    print(
        f"{result.case_id:<22} {'pass' if result.passed else 'fail':<5} {result.failed_check_names}"
    )

## Reading a failure: who owns it?

A failing check names a symptom. Assigning it to an owner takes one more step,
and the grounded result carries what is needed to do that.

| Owner | Typical signal | Example |
| --- | --- | --- |
| Input problem | The intake is thin or contradictory | One interest given, no values, and a generic pathway follows |
| Corpus gap | Retrieval was on topic but the material does not cover it | Handling criticism from extended family |
| Retrieval problem | The corpus covers it but the evidence came back off topic | Query blended too many needs into one vector |
| Generation problem | Good evidence, poor use of it | Ungrounded citation, leaked ids, prescriptive tone |
| Evaluation-rule problem | The pathway is good and the check is wrong | A rule flags a paraphrase that is genuinely personalized |

The last row is real. A check that fires on good output is a bug in the check,
and the fix belongs in the evaluation suite rather than in the prompt.

## The recorded baseline

Running the six real evaluation cases through the full pipeline gave:

| Metric | Value |
| --- | --- |
| Cases evaluated | 6 |
| Cases passed | 6 |
| Checks evaluated | 60 |
| Checks passed | 60 |
| Human review | Completed against the pathway review rubric |

Every automated check passed on every case, and a person still read all six
pathways. That combination is the point: the suite proves nothing was obviously
broken, and the review answers whether the pathways were any good.

## What this evaluation cannot tell you

* Lexical indicators can reward keyword echoing rather than real personalization.
* Id grounding proves provenance, not semantic support for the claim.
* Phrase scans miss paraphrases, so prescriptive advice can pass in softer words.
* Exact duplicate detection misses near-duplicates that read as repetition.
* Hard pass or fail checks carry no severity weighting, so a cosmetic failure and
  an ungrounded citation look the same in the totals.
* Human review remains necessary, which is why the rubric is a committed
  document and not an informal habit.

## Key takeaways

* Retrieval and pathway evaluation answer different questions and stay separate.
* Cases are saved fixtures, so results change only when the system changes.
* Ten deterministic checks cover structure, grounding, and tone; each returns a
  name, a boolean, and a reason.
* Case-level and check-level totals should both be read; they fail differently.
* Every check is cheap and shallow by design, and the suite's honest limits are
  documented rather than hidden.

## Next

Notebook 7 puts the service behind the two interfaces a person actually touches:
a Streamlit page and a small HTTP API.